# Gold: chamados_nota

Uma linha por chamado (131.838).

## O que ela faz
Junta, em uma tabela só, tudo que é preciso para estudar a **satisfação do cliente**: o chamado, a nota que ele deu, o motivo e quanto tempo ele esperou e foi atendido.

## Como funciona
Parte da tabela de **chamados** (uma linha por chamado) e junta:

| De onde vem | O que traz |
|---|---|
| Pesquisa | A nota (1 a 5), ou vazio se o cliente não respondeu |
| Motivos | Categoria (Cartão, ...) e criticidade (alta, média, baixa) |
| Chamados_hora | O mês do chamado, se foi abandono e os horários para corrigir os tempos |

No caminho, a tabela também:
- **corrige os tempos**: usa o maior valor entre o registrado e o calculado pelos horários, porque o registrado não passa de ~16 min (talvez porque o limite são 3 dígitos: 999s). Onde o registro passa de 10 s, ele bate 100% com os horários;
- **agrupa os tempos em faixas** de 5 minutos (teto de 60), para os gráficos;
- **confere** que nenhum chamado foi perdido ou duplicado (`assert` de 131.838 linhas).

## Colunas principais

| Coluna | O que é |
|---|---|
| `id_chamado`, `canal`, `motivo`, `categoria`, `criticidade` | Que chamado é e de que tipo |
| `resolvido` | Se o problema foi resolvido |
| `nota`, `tem_nota` | A nota e se o cliente respondeu |
| `espera_seg`, `atend_seg` | Tempo real de espera e de atendimento, em segundos (corrigidos) |
| `faixa_espera_min`, `faixa_atend_min` | Os tempos em faixas (0, 5, 10... minutos) |
| `mes` | Mês do chamado |
| `abandono` | Se o cliente desistiu na fila |

Para o tempo, use `espera_seg` e `atend_seg`, e **não** `tempo_espera` e `tempo_atendimento` (registro original, subestimado).

## Para fazer as análises

| Análise | Pergunta de negócio | Colunas |
|---|---|---|
| Nota × tempo de espera | A espera derruba a nota? | `nota`, `espera_seg` |
| Nota × tempo de atendimento | Atendimento longo derruba a nota? | `nota`, `atend_seg` |
| Nota × resolvido / não resolvido | Não resolver pesa mais que esperar? | `nota`, `resolvido` |
| Gráfico nota × faixa de espera | A partir de quantos minutos a nota despenca? (vira meta de espera) | `faixa_espera_min`, `nota` |
| Gráfico nota × faixa de atendimento | Idem para o atendimento | `faixa_atend_min`, `nota` |
| Nota ao longo do tempo | A satisfação caiu porque os bons diminuíram ou os ruins cresceram? Testa a tese de "fator externo" da Marketing | `mes`, `nota` |
| Tipo de chamado ao longo do tempo | Os clientes passaram a ligar por motivos mais difíceis? | `mes`, `categoria` |

## O que já saiu desta tabela
O tempo que a V-Credit enxerga está muito abaixo do real:

| Canal | Atendimento registrado | Atendimento real | Espera registrada | Espera real |
|---|---|---|---|---|
| Atendimento inicial (nível 1) | 6,1 min | **16,1 min** | 5,8 min | 6,7 min |
| Atendimento especializado (nível 2) | 1,7 min | **38,7 min** | 5,7 min | **12,7 min** |

## Observações
- **Espera** só existe nos canais humanos (URA e chatbot não têm fila).
- **Pesquisa:** 802 chamados tinham duas respostas, sempre com a mesma nota; fica uma só.
- **Nota:** 73.383 chamados (56%) têm nota. Dos 3.408 abandonos, só 703 (21%) responderam, então a satisfação medida tende a ser otimista.
- **Sem data:** 199 chamados (removidos na silver por horário impossível) ficam fora dos gráficos por mês.

In [0]:
from pyspark.sql import functions as F

s = "rocket_lab.vcredit_silver"
g = "rocket_lab.vcredit_gold"
#spark.sql(f"CREATE SCHEMA IF NOT EXISTS {g}")
#print("schema pronto:", g)

In [0]:
LARGURA, TETO = 5, 60   # minutos (ajuste depois da célula de percentis)

ch = (spark.table(f"{s}.chamados")
      .select("id_chamado", "motivo", "canal", "resolvido", "tempo_espera", "tempo_atendimento"))

hr = (spark.table(f"{s}.chamados_hora")
      .select("id_chamado",
              F.col("hora_abertura_chamado").alias("abertura"),
              F.col("hora_inicio_atendimento").alias("inicio"),
              F.col("hora_finalizacao_atendimento").alias("fim"),
              F.col("sem_inicio_atendimento").alias("abandono")))

mo = (spark.table(f"{s}.motivos")
      .select(F.col("nome_motivo").alias("motivo"), "categoria", "criticidade"))

# Pesquisa: Uma linha por chamado
pe = (spark.table(f"{s}.pesquisa").groupBy("id_chamado")
        .agg(F.max("nota_atendimento").alias("nota"),
             F.min("nota_atendimento").alias("nota_min"),
             F.count("*").alias("qtd_respostas")))
print("Chamados com mais de uma resposta:", pe.filter("qtd_respostas > 1").count(),
      "| Com notas diferentes entre as respostas:", pe.filter("nota != nota_min").count())
pe = pe.select("id_chamado", "nota", "qtd_respostas")

humano = F.col("canal").isin("atendimento inicial", "atendimento especializado")
espera_calc = F.unix_timestamp("inicio") - F.unix_timestamp("abertura")
atend_calc = F.unix_timestamp("fim") - F.unix_timestamp("inicio")

def faixa_min(col):
    b = F.floor(F.col(col) / 60 / LARGURA) * LARGURA
    return F.when(F.col(col).isNotNull(), F.least(b, F.lit(TETO)))

f = (ch.join(hr, "id_chamado", "left")
       .join(pe, "id_chamado", "left")
       .join(mo, "motivo", "left")
       .withColumn("mes", F.trunc(F.to_date("abertura"), "month"))
       .withColumn("abandono", F.coalesce("abandono", F.lit(False)))
       .withColumn("qtd_respostas", F.coalesce("qtd_respostas", F.lit(0)))
       .withColumn("tem_nota", F.col("nota").isNotNull())
       # Duração final = maior valor entre o registrado e o calculado pelos horários
       .withColumn("espera_seg", F.when(humano, F.greatest(F.col("tempo_espera"), espera_calc)))
       .withColumn("atend_seg", F.greatest(F.col("tempo_atendimento"), atend_calc))
       .withColumn("espera_corrigida", humano & ~F.col("espera_seg").eqNullSafe(F.col("tempo_espera")))
       .withColumn("atend_corrigido", ~F.col("atend_seg").eqNullSafe(F.col("tempo_atendimento")))
       .withColumn("faixa_espera_min", faixa_min("espera_seg"))
       .withColumn("faixa_atend_min", faixa_min("atend_seg"))
       .drop("inicio", "fim"))

n_ch, n_f = ch.count(), f.count()
print("Chamados:", n_ch, "| Chamados_nota:", n_f)
assert n_f == n_ch, "o join duplicou ou perdeu chamados!"

(f.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
   .saveAsTable(f"{g}.chamados_nota"))

t = spark.table(f"{g}.chamados_nota")
print("Sem data:", t.filter("mes is null").count(),
      "| Com nota:", t.filter("tem_nota").count(),
      "| Sem categoria:", t.filter("categoria is null").count(),
      "| Abandonos:", t.filter("abandono").count(),
      "| Abandonos com nota:", t.filter("abandono and tem_nota").count())
print("Espera corrigida:", t.filter("espera_corrigida").count(),
      "| Atendimento corrigido:", t.filter("atend_corrigido").count())

# média registrada x média corrigida (só canais humanos)
th = t.filter("canal in ('atendimento inicial', 'atendimento especializado')")
display(th.groupBy("canal").agg(
    F.round(F.avg("tempo_espera") / 60, 1).alias("espera_media_registrada_min"),
    F.round(F.avg("espera_seg") / 60, 1).alias("espera_media_corrigida_min"),
    F.round(F.avg("tempo_atendimento") / 60, 1).alias("atend_medio_registrado_min"),
    F.round(F.avg("atend_seg") / 60, 1).alias("atend_medio_corrigido_min")))

In [0]:
from pyspark.sql import functions as F
g = "rocket_lab.vcredit_gold"

t = spark.table(f"{g}.chamados_nota").filter("tem_nota")
t_all = spark.table(f"{g}.chamados_nota").filter("mes is not null")

# 1) nota x tempo médio, resolvido e não resolvido
display(t.groupBy("nota", "resolvido").agg(
    F.count("*").alias("chamados"),
    F.round(F.avg("espera_seg") / 60, 1).alias("espera_media_min"),
    F.round(F.avg("atend_seg") / 60, 1).alias("atendimento_medio_min"))
  .orderBy("nota", "resolvido"))

# 2) nota x faixa de espera
display(t.filter("faixa_espera_min is not null")
          .groupBy("faixa_espera_min", "nota").count().orderBy("faixa_espera_min", "nota"))

# 3) nota média por faixa de espera (onde a nota despenca)
display(t.filter("faixa_espera_min is not null").groupBy("faixa_espera_min")
          .agg(F.round(F.avg("nota"), 2).alias("nota_media"), F.count("*").alias("chamados"))
          .orderBy("faixa_espera_min"))

# 4) nota x faixa de atendimento
display(t.filter("faixa_atend_min is not null")
          .groupBy("faixa_atend_min", "nota").count().orderBy("faixa_atend_min", "nota"))

# 5) nota ao longo do tempo
display(t_all.filter("tem_nota").groupBy("mes", "nota").count().orderBy("mes", "nota"))
display(t_all.groupBy("mes").agg(
    F.count("*").alias("chamados"),
    F.round(F.avg(F.col("tem_nota").cast("int")) * 100, 1).alias("taxa_resposta_pct"),
    F.round(F.avg("nota"), 2).alias("nota_media")).orderBy("mes"))

# 6) tipo de chamado ao longo do tempo
display(t_all.groupBy("mes", "categoria").count().orderBy("mes", "categoria"))